# 서식지 적합성 지도 처리 및 지역별 데이터 정리

이 노트북은 SDM.ipynb에서 생성된 GeoTIFF 파일들을 처리하여:
1. TIF 파일을 CSV로 변환
2. 지역(시군구)별로 데이터 분류
3. Latin Hypercube Sampling으로 각 지역당 400개 샘플링
4. Pickle 파일로 저장

## 필요 패키지
```bash
pip install rasterio geopandas pandas numpy scipy openpyxl
```

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path
from scipy.stats import qmc
import pickle
import warnings
warnings.filterwarnings('ignore')

print("패키지 로드 완료!")

## 1. 경로 설정

In [ ]:
# 경로 설정
base_path = Path('/Users/mkim/Library/CloudStorage/GoogleDrive-wuj2293@gmail.com/내 드라이브/Research/CA/Data')
output_path = base_path / 'outputs'
latlong_file = Path('/Users/mkim/Library/CloudStorage/GoogleDrive-wuj2293@gmail.com/내 드라이브/Research/CA_old/슈퍼컴CA_2025/data/latin/latlong_ex.xlsx')

# 출력 폴더 생성
csv_output_path = output_path / 'csv'
csv_output_path.mkdir(exist_ok=True)

regional_output_path = output_path / 'regional_latin'
regional_output_path.mkdir(exist_ok=True)

print(f"출력 경로: {output_path}")
print(f"CSV 저장 경로: {csv_output_path}")
print(f"지역별 데이터 저장 경로: {regional_output_path}")

## 2. TIF 파일을 CSV로 변환

In [ ]:
def tif_to_csv(tif_file, output_csv):
    """
    GeoTIFF 파일을 CSV로 변환 (x, y, value)
    
    Parameters:
    -----------
    tif_file : Path
        입력 GeoTIFF 파일 경로
    output_csv : Path
        출력 CSV 파일 경로
    """
    with rasterio.open(tif_file) as src:
        # 래스터 데이터 읽기
        data = src.read(1)
        transform = src.transform
        
        # 유효한 데이터만 추출 (NoData 제외)
        rows, cols = np.where((~np.isnan(data)) & (data > -9000))
        
        # 픽셀 좌표를 지리 좌표로 변환
        xs, ys = rasterio.transform.xy(transform, rows, cols)
        values = data[rows, cols]
        
        # DataFrame 생성
        df = pd.DataFrame({
            'x': xs,
            'y': ys,
            tif_file.stem.replace('suitability_map_', ''): values
        })
        
        # CSV 저장
        df.to_csv(output_csv, index=False)
        
        return df

# 테스트: 현재 시나리오 변환
current_tif = output_path / 'suitability_map_current.tif'
if current_tif.exists():
    test_df = tif_to_csv(current_tif, csv_output_path / 'current.csv')
    print(f"현재 시나리오 변환 완료: {len(test_df)} 포인트")
    print(test_df.head())
else:
    print(f"파일을 찾을 수 없습니다: {current_tif}")

In [ ]:
# 모든 시나리오 파일 변환
scenarios = [
    'ssp126_2030', 'ssp126_2050', 'ssp126_2070', 'ssp126_2090',
    'ssp245_2030', 'ssp245_2050', 'ssp245_2070', 'ssp245_2090',
    'ssp585_2030', 'ssp585_2050', 'ssp585_2070', 'ssp585_2090'
]

csv_files = {}

for scenario in scenarios:
    tif_file = output_path / f'suitability_map_{scenario}.tif'
    
    if tif_file.exists():
        csv_file = csv_output_path / f'{scenario}.csv'
        df = tif_to_csv(tif_file, csv_file)
        csv_files[scenario] = csv_file
        print(f"✓ {scenario}: {len(df)} 포인트")
    else:
        print(f"✗ {scenario}: 파일 없음")

print(f"\n총 {len(csv_files)}개 시나리오 변환 완료!")

## 3. 위경도-행정구역 매칭 데이터 로드

In [ ]:
# 위경도-행정구역 데이터 로드
# 주의: latlong_ex.xlsx 파일의 실제 경로를 확인하세요
try:
    latlong = pd.read_excel(latlong_file)
    latlong_dropped = latlong.drop(columns=['OBJECTID'], errors='ignore')
    
    # Gwangju-si, Goseong-gun, Inje-gun 중복 지역 처리
    latlong_dropped.loc[(latlong_dropped['SIG_ENG_NM'] == 'Gwangju-si') & (latlong_dropped['y'] <= 36.0), 'SIG_ENG_NM'] = 'Gwangju-si1'
    latlong_dropped.loc[(latlong_dropped['SIG_ENG_NM'] == 'Gwangju-si') & (latlong_dropped['y'] > 36.0), 'SIG_ENG_NM'] = 'Gwangju-si2'
    latlong_dropped.loc[(latlong_dropped['SIG_ENG_NM'] == 'Goseong-gun') & (latlong_dropped['y'] <= 37.0), 'SIG_ENG_NM'] = 'Goseong-gun1'
    latlong_dropped.loc[(latlong_dropped['SIG_ENG_NM'] == 'Goseong-gun') & (latlong_dropped['y'] > 37.0), 'SIG_ENG_NM'] = 'Goseong-gun2'
    latlong_dropped.loc[(latlong_dropped['SIG_ENG_NM'] == 'Inje-gun') & (latlong_dropped['y'] > 37.0), 'SIG_ENG_NM'] = 'Inje-gun1'
    
    # 좌표 반올림 (소수점 3자리)
    latlong_round = latlong_dropped.round({'x': 3, 'y': 3})
    
    print(f"위경도 데이터 로드 완료: {len(latlong_round)} 포인트")
    print(f"지역 수: {latlong_round['SIG_ENG_NM'].nunique()}")
    print(f"\n지역 목록:\n{latlong_round['SIG_ENG_NM'].value_counts().head(10)}")
    
except FileNotFoundError:
    print(f"오류: {latlong_file} 파일을 찾을 수 없습니다.")
    print("파일 경로를 확인하거나 해당 파일을 준비해주세요.")
    latlong_round = None

## 4. 지역별 데이터 분류 및 Latin Hypercube Sampling

In [ ]:
if latlong_round is not None:
    # 결과 저장용 딕셔너리
    maxent_datas = {}  # 전체 데이터
    maxent_datas_sampling = {}  # 샘플링된 데이터
    
    # Latin Hypercube Sampling 설정
    np.random.seed(0)
    num_samples = 400
    sampler = qmc.LatinHypercube(d=2)
    sample_indices = sampler.random(n=num_samples)
    
    # 각 시나리오별 처리
    for scenario, csv_file in csv_files.items():
        print(f"\n처리 중: {scenario}")
        
        # CSV 로드
        maxent_df = pd.read_csv(csv_file)
        
        # 좌표 반올림 (소수점 3자리)
        maxent_round = maxent_df.round({'x': 3, 'y': 3})
        
        # 위경도 데이터와 병합 (지역 정보 추가)
        merged = pd.merge(maxent_round, latlong_round, how='inner', on=['x', 'y'])
        
        print(f"  매칭된 포인트: {len(merged)}")
        
        # 지역별로 분류
        local_values = merged['SIG_ENG_NM'].drop_duplicates().tolist()
        
        maxent_data = {}
        maxent_data_sampling = {}
        
        for local_value in local_values:
            try:
                # 해당 지역 데이터 필터링
                filtered_df = merged[merged['SIG_ENG_NM'] == local_value]
                maxent_data[local_value] = filtered_df
                
                # Latin Hypercube Sampling
                if len(filtered_df) > 0:
                    sample_indices0 = (sample_indices * len(filtered_df)).astype(int)
                    sample_indices0 = sample_indices0[:, 0]
                    # 인덱스 범위 제한
                    sample_indices0 = np.clip(sample_indices0, 0, len(filtered_df) - 1)
                    sampled_data = filtered_df.iloc[sample_indices0]
                    maxent_data_sampling[local_value] = sampled_data
                
            except Exception as e:
                print(f"  경고: {local_value} 처리 중 오류 - {e}")
        
        maxent_datas[scenario] = maxent_data
        maxent_datas_sampling[scenario] = maxent_data_sampling
        
        print(f"  지역 수: {len(maxent_data)}")
    
    print(f"\n✓ 총 {len(maxent_datas)}개 시나리오 처리 완료!")
else:
    print("위경도 데이터가 없어 지역별 분류를 건너뜁니다.")

## 5. 결과 저장 (Pickle)

In [ ]:
if latlong_round is not None and len(maxent_datas) > 0:
    # Pickle 파일로 저장
    all_data_file = regional_output_path / 'all.pkl'
    sampling_data_file = regional_output_path / 'sampling.pkl'
    local_index_file = regional_output_path / 'local_index.csv'
    
    with open(all_data_file, 'wb') as f:
        pickle.dump(maxent_datas, f)
    print(f"✓ 전체 데이터 저장: {all_data_file}")
    
    with open(sampling_data_file, 'wb') as f:
        pickle.dump(maxent_datas_sampling, f)
    print(f"✓ 샘플링 데이터 저장: {sampling_data_file}")
    
    # 지역 목록 저장
    local_list = pd.Series(list(maxent_datas[list(maxent_datas.keys())[0]].keys()))
    local_list.to_csv(local_index_file, index=False, header=['SIG_ENG_NM'])
    print(f"✓ 지역 목록 저장: {local_index_file}")
    
    print(f"\n전체 지역 수: {len(local_list)}")
else:
    print("저장할 데이터가 없습니다.")

## 6. 결과 확인

In [ ]:
# 샘플 데이터 확인
if len(maxent_datas_sampling) > 0:
    # 예시: ssp126_2030의 Inje-gun1 데이터
    scenario_example = 'ssp126_2030'
    region_example = 'Inje-gun1'
    
    if scenario_example in maxent_datas_sampling and region_example in maxent_datas_sampling[scenario_example]:
        sample_data = maxent_datas_sampling[scenario_example][region_example]
        
        print(f"\n=== {scenario_example} - {region_example} 샘플링 데이터 ===")
        print(f"샘플 수: {len(sample_data)}")
        print(sample_data.head(10))
        
        # 시각화
        plt.figure(figsize=(10, 6))
        plt.scatter(sample_data['x'], sample_data['y'], 
                   c=sample_data.iloc[:, 2], cmap='YlOrRd', 
                   alpha=0.6, edgecolors='black', linewidth=0.5)
        plt.colorbar(label='Habitat Suitability')
        plt.xlabel('Longitude (x)', fontsize=12)
        plt.ylabel('Latitude (y)', fontsize=12)
        plt.title(f'{scenario_example} - {region_example} (Latin Hypercube Sampling)', 
                 fontsize=14, fontweight='bold')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print(f"\n{scenario_example} 또는 {region_example}을(를) 찾을 수 없습니다.")
        print(f"사용 가능한 시나리오: {list(maxent_datas_sampling.keys())}")
        if len(maxent_datas_sampling) > 0:
            first_scenario = list(maxent_datas_sampling.keys())[0]
            print(f"사용 가능한 지역 (예: {first_scenario}): {list(maxent_datas_sampling[first_scenario].keys())[:5]}...")

## 7. 데이터 로드 예제 (나중에 사용)

In [ ]:
# 저장된 데이터를 다시 불러오는 방법
def load_regional_data():
    """
    저장된 지역별 데이터를 로드
    """
    all_data_file = regional_output_path / 'all.pkl'
    sampling_data_file = regional_output_path / 'sampling.pkl'
    local_index_file = regional_output_path / 'local_index.csv'
    
    with open(all_data_file, 'rb') as f:
        all_data = pickle.load(f)
    
    with open(sampling_data_file, 'rb') as f:
        sampling_data = pickle.load(f)
    
    local_list = pd.read_csv(local_index_file)
    
    return all_data, sampling_data, local_list

# 사용 예제 (주석 해제 후 실행)
# all_data, sampling_data, local_list = load_regional_data()
# print(f"시나리오 수: {len(all_data)}")
# print(f"지역 수: {len(local_list)}")

## 요약

이 노트북은 다음 작업을 수행합니다:

1. **TIF → CSV 변환**: SDM.ipynb의 GeoTIFF 결과를 CSV로 변환
2. **지역 매칭**: 위경도 데이터를 사용하여 각 포인트에 행정구역 정보 추가
3. **지역별 분류**: 각 시나리오를 지역별로 분류
4. **Latin Hypercube Sampling**: 각 지역당 400개 포인트를 균등하게 샘플링
5. **저장**: Pickle 형식으로 저장하여 CA 모델에서 사용 가능

### 출력 파일:
- `Data/outputs/csv/*.csv`: 각 시나리오의 전체 포인트 데이터
- `Data/outputs/regional_latin/all.pkl`: 지역별 전체 데이터
- `Data/outputs/regional_latin/sampling.pkl`: 지역별 샘플링 데이터 (400개)
- `Data/outputs/regional_latin/local_index.csv`: 지역 목록

### 다음 단계:
CA 모델에서 `sampling.pkl` 파일을 로드하여 사용